# 01. Sepsis Cases Event Log - Exploratory Data Analysis

This notebook loads the Sepsis Cases event log and performs initial exploration.

**Dataset**: Sepsis Cases Event Log  
**Source**: https://data.4tu.nl/articles/dataset/Sepsis_Cases_-_Event_Log/12707639  
**Expected**: ~1,050 cases, ~15,214 events, 16 activities

In [ ]:
import pm4py
import pandas as pd
import matplotlib.pyplot as plt

# Load configuration (paths from .env)
from config import SEPSIS_PATH, FIGURES_PATH, RESULTS_PATH

print(f"PM4Py version: {pm4py.__version__}")
print(f"Data path: {SEPSIS_PATH}")

## 1. Load XES Event Log

In [ ]:
# Load XES file
log = pm4py.read_xes(str(SEPSIS_PATH))
print(f"Loaded event log from: {SEPSIS_PATH}")
print(f"Type: {type(log)}")

## 2. Basic Statistics

In [ ]:
# Basic counts
n_cases = len(log['case:concept:name'].unique())
n_events = len(log)
n_activities = len(log['concept:name'].unique())

print("="*50)
print("BASELINE STATISTICS")
print("="*50)
print(f"Number of cases:      {n_cases:,} (expected: 1,050)")
print(f"Number of events:     {n_events:,} (expected: 15,214)")
print(f"Number of activities: {n_activities} (expected: 16)")
print(f"Avg events per case:  {n_events/n_cases:.1f}")

In [ ]:
# Show dataframe structure
print("\nDataFrame columns:")
print(log.columns.tolist())
print(f"\nShape: {log.shape}")

In [ ]:
# Preview first rows
log.head(10)

## 3. Activity Analysis

In [ ]:
# Activity frequency
activity_counts = log['concept:name'].value_counts()
print("Activity frequencies:")
print(activity_counts)

In [ ]:
# Visualize activity distribution
fig, ax = plt.subplots(figsize=(12, 6))
activity_counts.plot(kind='barh', ax=ax)
ax.set_xlabel('Frequency')
ax.set_ylabel('Activity')
ax.set_title('Activity Frequency Distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'sepsis_activity_distribution.png', dpi=150)
plt.show()

## 4. Case Duration Analysis

In [ ]:
# Calculate case durations
case_durations = log.groupby('case:concept:name').agg(
    start_time=('time:timestamp', 'min'),
    end_time=('time:timestamp', 'max'),
    n_events=('concept:name', 'count')
)
case_durations['duration'] = case_durations['end_time'] - case_durations['start_time']
case_durations['duration_hours'] = case_durations['duration'].dt.total_seconds() / 3600
case_durations['duration_days'] = case_durations['duration_hours'] / 24

print("Case duration statistics (days):")
print(case_durations['duration_days'].describe())

In [ ]:
# Visualize case duration distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration histogram
axes[0].hist(case_durations['duration_days'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Duration (days)')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('Case Duration Distribution')
axes[0].axvline(case_durations['duration_days'].median(), color='red', linestyle='--', 
                label=f"Median: {case_durations['duration_days'].median():.1f} days")
axes[0].legend()

# Events per case histogram
axes[1].hist(case_durations['n_events'], bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of events')
axes[1].set_ylabel('Number of cases')
axes[1].set_title('Events per Case Distribution')
axes[1].axvline(case_durations['n_events'].median(), color='red', linestyle='--', 
                label=f"Median: {case_durations['n_events'].median():.0f} events")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'sepsis_case_distributions.png', dpi=150)
plt.show()

## 5. Clinical Attributes Analysis

In [ ]:
# Check for clinical attributes (SIRS criteria, CRP, Lactic Acid)
clinical_attrs = [
    'Age', 'Diagnose', 'DisfuncOrg', 'Hypotensie', 'Hypoxie', 
    'InfectionSuspected', 'Infusion', 'Oligurie',
    'SIRSCritHeartRate', 'SIRSCritLeucos', 'SIRSCritTachypnea',
    'SIRSCritTemperature', 'SIRSCriteria2OrMore', 
    'Leucocytes', 'CRP', 'LacticAcid'
]

# Find which clinical attributes exist in the log
existing_attrs = [attr for attr in clinical_attrs if attr in log.columns]
print(f"Found {len(existing_attrs)} clinical attributes:")
for attr in existing_attrs:
    non_null = log[attr].notna().sum()
    print(f"  - {attr}: {non_null:,} non-null values")

In [ ]:
# Analyze numeric clinical attributes if present
numeric_attrs = ['Leucocytes', 'CRP', 'LacticAcid']
for attr in numeric_attrs:
    if attr in log.columns:
        values = pd.to_numeric(log[attr], errors='coerce').dropna()
        if len(values) > 0:
            print(f"\n{attr} statistics:")
            print(values.describe())

## 6. Process Variants

In [ ]:
# Get process variants using PM4Py
variants = pm4py.get_variants(log)
print(f"Number of unique process variants: {len(variants)}")

# Sort by frequency
variants_sorted = sorted(variants.items(), key=lambda x: x[1], reverse=True)

print(f"\nTop 10 most frequent variants:")
for i, (variant, count) in enumerate(variants_sorted[:10], 1):
    coverage = count / n_cases * 100
    variant_str = str(variant)[:80]
    print(f"{i:2d}. [{count:4d} cases, {coverage:5.1f}%] {variant_str}{'...' if len(str(variant)) > 80 else ''}")

In [ ]:
# Variant coverage analysis
cumulative_coverage = 0
for i, (variant, count) in enumerate(variants_sorted, 1):
    cumulative_coverage += count
    if cumulative_coverage / n_cases >= 0.8:
        print(f"80% of cases covered by top {i} variants")
        break

cumulative_coverage = 0
for i, (variant, count) in enumerate(variants_sorted, 1):
    cumulative_coverage += count
    if cumulative_coverage / n_cases >= 0.5:
        print(f"50% of cases covered by top {i} variants")
        break

## 7. Start and End Activities

In [ ]:
# Start activities
start_activities = pm4py.get_start_activities(log)
print("Start activities:")
for activity, count in sorted(start_activities.items(), key=lambda x: -x[1]):
    print(f"  {activity}: {count} ({count/n_cases*100:.1f}%)")

In [ ]:
# End activities
end_activities = pm4py.get_end_activities(log)
print("End activities:")
for activity, count in sorted(end_activities.items(), key=lambda x: -x[1]):
    print(f"  {activity}: {count} ({count/n_cases*100:.1f}%)")

## 8. Directly-Follows Graph (Preview)

In [ ]:
# Generate DFG for quick visualization
dfg, start_acts, end_acts = pm4py.discover_dfg(log)

print(f"DFG has {len(dfg)} edges")
print("\nTop 10 most frequent transitions:")
for (src, tgt), count in sorted(dfg.items(), key=lambda x: -x[1])[:10]:
    print(f"  {src} -> {tgt}: {count}")

In [ ]:
# Save DFG visualization
pm4py.save_vis_dfg(dfg, start_acts, end_acts, str(FIGURES_PATH / 'sepsis_dfg.png'))
print(f"DFG saved to {FIGURES_PATH / 'sepsis_dfg.png'}")

## 9. Summary Statistics Export

In [ ]:
# Create summary statistics dictionary
summary = {
    'n_cases': n_cases,
    'n_events': n_events,
    'n_activities': n_activities,
    'n_variants': len(variants),
    'avg_events_per_case': n_events / n_cases,
    'median_duration_days': case_durations['duration_days'].median(),
    'mean_duration_days': case_durations['duration_days'].mean(),
    'dfg_edges': len(dfg)
}

# Save to CSV
pd.DataFrame([summary]).to_csv(RESULTS_PATH / 'sepsis_summary_stats.csv', index=False)
print(f"Summary statistics saved to {RESULTS_PATH / 'sepsis_summary_stats.csv'}")

# Display summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

## Next Steps

1. **02_sepsis_discovery.ipynb**: Apply Inductive Miner and Heuristic Miner
2. **03_sepsis_conformance.ipynb**: Conformance checking against discovered models
3. **04_sepsis_performance.ipynb**: Bottleneck analysis and outcome correlation